# CIC-IDS 4-Class Classification Notebook
**Classes:** Benign, BruteForce, DoS, Botnet  
**Improved version** with each model in its own cell for easy monitoring on Colab.

---
### ▶ Run cells top-to-bottom. Each model cell is independent — if one crashes you can re-run just that cell.

## 📦 Cell 1 — Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    roc_auc_score, ConfusionMatrixDisplay, accuracy_score,
    precision_score, recall_score
)
from sklearn.feature_selection import mutual_info_classif, VarianceThreshold

# Models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb

# Imbalance handling
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.combine import SMOTETomek
from collections import Counter

print('=' * 70)
print('CIC-IDS 4-CLASS CLASSIFICATION — FRAGMENTED VERSION')
print('All imports loaded successfully ✓')
print('=' * 70)

CIC-IDS 4-CLASS CLASSIFICATION — FRAGMENTED VERSION
All imports loaded successfully ✓


## 📂 Cell 2 — Load Data

In [ ]:
print('Loading data...')
t0 = time.time()

df = pd.read_parquet('part_01.parquet')

# Drop original Label column if it exists (use ClassLabel only)
if 'Label' in df.columns:
    df = df.drop(columns='Label')
    print('  Dropped original Label column')

print(f'Raw shape: {df.shape}')
print(f'\nClasses found:\n{df["ClassLabel"].value_counts()}')
print(f'\nClass % distribution:')
print((df['ClassLabel'].value_counts(normalize=True) * 100).round(2))
print(f'\n✓ Data loaded in {time.time()-t0:.1f}s')

Loading data...
  Dropped original Label column
Raw shape: (916234, 58)

Classes found:
ClassLabel
Benign        718619
DoS           163208
BruteForce     19810
Botnet         14597
Name: count, dtype: int64

Class % distribution:
ClassLabel
Benign        78.43
DoS           17.81
BruteForce     2.16
Botnet         1.59
Name: proportion, dtype: float64

✓ Data loaded in 2.5s


## 🔍 Cell 3 — Data Quality Audit

In [ ]:
print('=' * 70)
print('DATA QUALITY AUDIT')
print('=' * 70)

X = df.drop(columns=['ClassLabel'])
y = df['ClassLabel']

print(f'Features: {X.shape[1]}  |  Rows: {X.shape[0]}')

# --- Infinity values ---
inf_cols = X.columns[np.isinf(X).any()]
print(f'\nInfinity columns: {len(inf_cols)}')
if len(inf_cols):
    print(' ', inf_cols.tolist())
    X = X.replace([np.inf, -np.inf], np.nan)
    print('  ✓ Replaced with NaN')

# --- NaN values ---
nan_cols = X.columns[X.isna().any()]
print(f'\nNaN columns: {len(nan_cols)}')
if len(nan_cols):
    X = X.fillna(X.median())
    print('  ✓ Filled with column median')

# --- Constant columns ---
constant_cols = X.columns[X.nunique() == 1]
print(f'\nConstant (zero-variance) columns: {len(constant_cols)}')
if len(constant_cols):
    print(' ', constant_cols.tolist())
    X = X.drop(columns=constant_cols)
    print(f'  ✓ Dropped {len(constant_cols)} constant columns')

# --- Duplicate columns (vectorised — avoids O(n²) column loop) ---
print('\nChecking for duplicate columns (vectorised)...')
# Hash each column; only compare within hash-groups
col_hashes = {col: hash(X[col].values.tobytes()) for col in X.columns}
seen = {}
duplicated_cols = []
for col, h in col_hashes.items():
    if h in seen:
        # Final byte-level equality check to avoid hash collisions
        if X[col].equals(X[seen[h]]):
            duplicated_cols.append(col)
    else:
        seen[h] = col

print(f'Duplicate columns: {len(duplicated_cols)}')
if duplicated_cols:
    X = X.drop(columns=duplicated_cols)
    print(f'  ✓ Dropped {len(duplicated_cols)} duplicate columns')

print(f'\n✓ Final feature shape after audit: {X.shape}')

DATA QUALITY AUDIT
Features: 57  |  Rows: 916234

Infinity columns: 0

NaN columns: 0

Constant (zero-variance) columns: 0

Checking for duplicate columns (vectorised)...
Duplicate columns: 4
  ✓ Dropped 4 duplicate columns

✓ Final feature shape after audit: (916234, 53)


## ⚙️ Cell 4 — Feature Engineering

In [ ]:
print('=' * 70)
print('FEATURE ENGINEERING')
print('=' * 70)

feature_eng_count = 0

if 'Fwd Packets/s' in X.columns and 'Bwd Packets/s' in X.columns:
    X['Fwd_Bwd_Packet_Ratio'] = X['Fwd Packets/s'] / (X['Bwd Packets/s'] + 1e-10)
    feature_eng_count += 1
    print('  ✓ Fwd_Bwd_Packet_Ratio')

if 'Total Fwd Packets' in X.columns and 'Total Backward Packets' in X.columns:
    X['Packet_Direction_Ratio'] = X['Total Fwd Packets'] / (X['Total Backward Packets'] + 1e-10)
    feature_eng_count += 1
    print('  ✓ Packet_Direction_Ratio')

if all(c in X.columns for c in ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets']):
    X['Avg_Duration_Per_Packet'] = X['Flow Duration'] / (
        X['Total Fwd Packets'] + X['Total Backward Packets'] + 1e-10
    )
    feature_eng_count += 1
    print('  ✓ Avg_Duration_Per_Packet')

length_cols = [c for c in X.columns if 'Packet Length' in c]
if len(length_cols) >= 2:
    X['Packet_Length_Variance_Custom'] = X[length_cols].var(axis=1)
    feature_eng_count += 1
    print('  ✓ Packet_Length_Variance_Custom')

print(f'\nCreated {feature_eng_count} engineered features')
print(f'Total features now: {X.shape[1]}')

FEATURE ENGINEERING
  ✓ Fwd_Bwd_Packet_Ratio
  ✓ Packet_Direction_Ratio
  ✓ Avg_Duration_Per_Packet
  ✓ Packet_Length_Variance_Custom

Created 4 engineered features
Total features now: 57


## 📊 Cell 5 — Outlier Capping (IQR)

In [ ]:
print('=' * 70)
print('OUTLIER CAPPING (3 × IQR)')
print('=' * 70)

t0 = time.time()
Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)
IQR = Q3 - Q1

outliers = ((X < (Q1 - 3 * IQR)) | (X > (Q3 + 3 * IQR))).sum()
top_outlier_cols = outliers[outliers > 0].sort_values(ascending=False).head(10)
print(f'Top 10 columns with most outliers (3×IQR):')
print(top_outlier_cols)

# Vectorised clip — much faster than per-column loop
lower_bounds = Q1 - 3 * IQR
upper_bounds = Q3 + 3 * IQR
X = X.clip(lower=lower_bounds, upper=upper_bounds, axis=1)

print(f'\n✓ Outliers capped in {time.time()-t0:.1f}s')

OUTLIER CAPPING (3 × IQR)
Top 10 columns with most outliers (3×IQR):
Fwd Seg Size Min      346023
Fwd IAT Min           204054
Idle Max              192840
Idle Mean             192840
Idle Min              192840
Fwd IAT Std           189209
Flow IAT Min          188512
Init Bwd Win Bytes    185321
Bwd IAT Min           182315
Bwd Packets/s         179272
dtype: int64

✓ Outliers capped in 3.8s


## 🎯 Cell 6 — Feature Selection

In [ ]:
print('=' * 70)
print('FEATURE SELECTION')
print('=' * 70)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f'Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Step 1: Variance threshold
selector_variance = VarianceThreshold(threshold=0.01)
selector_variance.fit(X)
selected_features = X.columns[selector_variance.get_support()].tolist()
print(f'\nVariance threshold: {len(selected_features)} / {X.shape[1]} features kept')

# Step 2: Mutual Information (top 50)
print('\nCalculating Mutual Information (this may take a minute)...')
t0 = time.time()
mi_scores = mutual_info_classif(X[selected_features], y_encoded, random_state=42)
mi_df = pd.DataFrame({'feature': selected_features, 'mi_score': mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=False)
top_mi_features = mi_df.head(50)['feature'].tolist()
print(f'✓ MI done in {time.time()-t0:.1f}s  |  Top 50 features selected')
print(f'\nTop 10 by MI:')
print(mi_df.head(10).to_string(index=False))

# Step 3: Remove highly correlated (>0.95)
print('\nRemoving highly correlated features (threshold=0.95)...')
corr_matrix = X[top_mi_features].corr().abs()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper_triangle.columns if any(upper_triangle[col] > 0.95)]
final_features = [f for f in top_mi_features if f not in to_drop]
print(f'Dropped {len(to_drop)} correlated  |  Final feature set: {len(final_features)}')

X_selected = X[final_features].copy()
print(f'\n✓ X_selected shape: {X_selected.shape}')

FEATURE SELECTION
Label encoding: {'Benign': np.int64(0), 'Botnet': np.int64(1), 'BruteForce': np.int64(2), 'DoS': np.int64(3)}

Variance threshold: 45 / 57 features kept

Calculating Mutual Information (this may take a minute)...
✓ MI done in 465.1s  |  Top 50 features selected

Top 10 by MI:
                      feature  mi_score
        Fwd Packet Length Max  0.406366
Packet_Length_Variance_Custom  0.394069
        Bwd Packet Length Max  0.392207
       Fwd Packet Length Mean  0.387289
            Packet Length Max  0.386041
            Subflow Fwd Bytes  0.379446
            Packet Length Std  0.379379
     Fwd Packets Length Total  0.379011
       Bwd Packet Length Mean  0.376480
              Avg Packet Size  0.375651

Removing highly correlated features (threshold=0.95)...
Dropped 17 correlated  |  Final feature set: 28

✓ X_selected shape: (916234, 28)


## ✂️ Cell 7 — Train/Test Split & Scaling

In [ ]:
print('=' * 70)
print('TRAIN/TEST SPLIT & SCALING')
print('=' * 70)

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'\nTrain class distribution:\n{y_train.value_counts()}')
print(f'\nTest class distribution:\n{y_test.value_counts()}')

# Scale
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Encode labels
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

print('\n✓ RobustScaler fitted and applied')
print('✓ Labels encoded')

TRAIN/TEST SPLIT & SCALING
Train: (732987, 28)  |  Test: (183247, 28)

Train class distribution:
ClassLabel
Benign        574895
DoS           130566
BruteForce     15848
Botnet         11678
Name: count, dtype: int64

Test class distribution:
ClassLabel
Benign        143724
DoS            32642
BruteForce      3962
Botnet          2919
Name: count, dtype: int64

✓ RobustScaler fitted and applied
✓ Labels encoded


## ⚖️ Cell 8 — Resampling (SMOTE variants)

In [ ]:
print('=' * 70)
print('CLASS IMBALANCE HANDLING')
print('=' * 70)
print(f'Original distribution: {Counter(y_train_enc)}')

# SMOTE
print('\nApplying SMOTE...')
t0 = time.time()
smote = SMOTE(random_state=42, k_neighbors=5)
X_smote, y_smote = smote.fit_resample(X_train_scaled, y_train_enc)
print(f'  ✓ Done in {time.time()-t0:.1f}s  |  {Counter(y_smote)}')

# BorderlineSMOTE
print('\nApplying BorderlineSMOTE...')
t0 = time.time()
bl_smote = BorderlineSMOTE(random_state=42, k_neighbors=5)
X_blsmote, y_blsmote = bl_smote.fit_resample(X_train_scaled, y_train_enc)
print(f'  ✓ Done in {time.time()-t0:.1f}s  |  {Counter(y_blsmote)}')


# Bundle strategies for easy iteration later
strategies = {
    'NoResample':      (X_train_scaled, y_train_enc),
    'SMOTE':           (X_smote,    y_smote),
    'BorderlineSMOTE': (X_blsmote,  y_blsmote),
}
print('\n✓ All resampling strategies ready')

CLASS IMBALANCE HANDLING
Original distribution: Counter({np.int64(0): 574895, np.int64(3): 130566, np.int64(2): 15848, np.int64(1): 11678})

Applying SMOTE...
  ✓ Done in 116.4s  |  Counter({np.int64(0): 574895, np.int64(3): 574895, np.int64(2): 574895, np.int64(1): 574895})

Applying BorderlineSMOTE...
  ✓ Done in 753.1s  |  Counter({np.int64(0): 574895, np.int64(3): 574895, np.int64(2): 574895, np.int64(1): 574895})

✓ All resampling strategies ready


## 🛠️ Cell 9 — Shared Evaluation Helper

In [ ]:
# Shared dict to collect results across model cells
results = {}

def evaluate_model(model, X_tr, y_tr, X_te, y_te, label):
    """Train model, evaluate on test set, store results."""
    print(f'  Training {label}...', end=' ', flush=True)
    t0 = time.time()
    model.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    print(f'done in {elapsed:.1f}s')

    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te) if hasattr(model, 'predict_proba') else None

    acc  = accuracy_score(y_te, y_pred)
    p_m  = precision_score(y_te, y_pred, average='macro',    zero_division=0)
    p_w  = precision_score(y_te, y_pred, average='weighted', zero_division=0)
    r_m  = recall_score(y_te, y_pred, average='macro',       zero_division=0)
    r_w  = recall_score(y_te, y_pred, average='weighted',    zero_division=0)
    f1_m = f1_score(y_te, y_pred, average='macro',           zero_division=0)
    f1_w = f1_score(y_te, y_pred, average='weighted',        zero_division=0)

    try:
        auc_m = roc_auc_score(y_te, y_proba, multi_class='ovr', average='macro')
        auc_w = roc_auc_score(y_te, y_proba, multi_class='ovr', average='weighted')
    except Exception:
        auc_m = auc_w = None

    cm = confusion_matrix(y_te, y_pred)

    print(f'  Accuracy={acc:.4f}  F1_macro={f1_m:.4f}  F1_weighted={f1_w:.4f}'
          + (f'  AUC={auc_m:.4f}' if auc_m else ''))
    print(classification_report(y_te, y_pred, target_names=le.classes_, zero_division=0))

    results[label] = dict(
        model=model, accuracy=acc,
        precision_macro=p_m, precision_weighted=p_w,
        recall_macro=r_m,    recall_weighted=r_w,
        f1_macro=f1_m,       f1_weighted=f1_w,
        roc_auc_macro=auc_m, roc_auc_weighted=auc_w,
        confusion_matrix=cm, y_pred=y_pred, y_pred_proba=y_proba
    )
    return results[label]

print('✓ evaluate_model() helper ready')
print('✓ results dict initialised')

✓ evaluate_model() helper ready
✓ results dict initialised


---
## 🌲 Cell 10 — Random Forest (all strategies)

In [ ]:
print('=' * 70)
print('RANDOM FOREST')
print('=' * 70)

for strategy_name, (X_tr, y_tr) in strategies.items():
    label = f'RandomForest + {strategy_name}'
    model = RandomForestClassifier(
        n_estimators=100, max_depth=20, min_samples_split=5,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    evaluate_model(model, X_tr, y_tr, X_test_scaled, y_test_enc, label)

print('\n✓ Random Forest — all strategies done')

RANDOM FOREST
  Training RandomForest + NoResample... done in 356.9s
  Accuracy=0.9550  F1_macro=0.8421  F1_weighted=0.9636  AUC=0.9872
              precision    recall  f1-score   support

      Benign       0.99      0.95      0.97    143724
      Botnet       0.99      0.99      0.99      2919
  BruteForce       0.29      0.70      0.41      3962
         DoS       1.00      1.00      1.00     32642

    accuracy                           0.96    183247
   macro avg       0.82      0.91      0.84    183247
weighted avg       0.98      0.96      0.96    183247

  Training RandomForest + SMOTE... done in 1124.4s
  Accuracy=0.9453  F1_macro=0.8304  F1_weighted=0.9571  AUC=0.9868
              precision    recall  f1-score   support

      Benign       0.99      0.94      0.96    143724
      Botnet       0.98      1.00      0.99      2919
  BruteForce       0.25      0.72      0.38      3962
         DoS       0.99      1.00      0.99     32642

    accuracy                           

## ⚡ Cell 11 — XGBoost (all strategies)

In [ ]:
print('=' * 70)
print('XGBOOST')
print('=' * 70)

n_classes = len(le.classes_)  # auto-detected — no hard-coded 4

for strategy_name, (X_tr, y_tr) in strategies.items():
    label = f'XGBoost + {strategy_name}'
    model = xgb.XGBClassifier(
        n_estimators=100, max_depth=10, learning_rate=0.1,
        objective='multi:softprob',   # softprob gives probabilities for AUC
        num_class=n_classes,
        eval_metric='mlogloss',
        use_label_encoder=False,
        random_state=42, n_jobs=-1
    )
    evaluate_model(model, X_tr, y_tr, X_test_scaled, y_test_enc, label)

print('\n✓ XGBoost — all strategies done')

XGBOOST
  Training XGBoost + NoResample... done in 66.1s
  Accuracy=0.9889  F1_macro=0.9184  F1_weighted=0.9874  AUC=0.9896
              precision    recall  f1-score   support

      Benign       0.99      1.00      0.99    143724
      Botnet       1.00      0.99      0.99      2919
  BruteForce       0.97      0.53      0.69      3962
         DoS       1.00      1.00      1.00     32642

    accuracy                           0.99    183247
   macro avg       0.99      0.88      0.92    183247
weighted avg       0.99      0.99      0.99    183247

  Training XGBoost + SMOTE... done in 181.1s
  Accuracy=0.9516  F1_macro=0.8385  F1_weighted=0.9617  AUC=0.9871
              precision    recall  f1-score   support

      Benign       0.99      0.95      0.97    143724
      Botnet       0.99      1.00      0.99      2919
  BruteForce       0.27      0.72      0.39      3962
         DoS       1.00      1.00      1.00     32642

    accuracy                           0.95    183247
   

## 💡 Cell 12 — LightGBM (all strategies)

In [ ]:
print('=' * 70)
print('LIGHTGBM')
print('=' * 70)

n_classes = len(le.classes_)

for strategy_name, (X_tr, y_tr) in strategies.items():
    label = f'LightGBM + {strategy_name}'
    model = lgb.LGBMClassifier(
        n_estimators=100, max_depth=10, learning_rate=0.1,
        objective='multiclass',
        num_class=n_classes,
        class_weight='balanced',
        random_state=42, n_jobs=-1, verbose=-1
    )
    evaluate_model(model, X_tr, y_tr, X_test_scaled, y_test_enc, label)

print('\n✓ LightGBM — all strategies done')

LIGHTGBM
  Training LightGBM + NoResample... done in 57.2s
  Accuracy=0.8944  F1_macro=0.7943  F1_weighted=0.9270  AUC=0.9893
              precision    recall  f1-score   support

      Benign       0.99      0.87      0.93    143724
      Botnet       0.99      1.00      0.99      2919
  BruteForce       0.15      0.85      0.26      3962
         DoS       0.99      1.00      1.00     32642

    accuracy                           0.89    183247
   macro avg       0.78      0.93      0.79    183247
weighted avg       0.98      0.89      0.93    183247

  Training LightGBM + SMOTE... done in 170.4s
  Accuracy=0.9582  F1_macro=0.8471  F1_weighted=0.9656  AUC=0.9866
              precision    recall  f1-score   support

      Benign       0.99      0.96      0.97    143724
      Botnet       0.99      1.00      0.99      2919
  BruteForce       0.31      0.69      0.43      3962
         DoS       0.99      1.00      1.00     32642

    accuracy                           0.96    183247


## 🌳 Cell 13 — Extra Trees (all strategies)

In [ ]:
print('=' * 70)
print('EXTRA TREES')
print('=' * 70)

for strategy_name, (X_tr, y_tr) in strategies.items():
    label = f'ExtraTrees + {strategy_name}'
    model = ExtraTreesClassifier(
        n_estimators=100, max_depth=20,
        class_weight='balanced',
        random_state=42, n_jobs=-1
    )
    evaluate_model(model, X_tr, y_tr, X_test_scaled, y_test_enc, label)

print('\n✓ Extra Trees — all strategies done')

EXTRA TREES
  Training ExtraTrees + NoResample... done in 171.8s
  Accuracy=0.9177  F1_macro=0.8006  F1_weighted=0.9378  AUC=0.9849
              precision    recall  f1-score   support

      Benign       0.99      0.90      0.95    143724
      Botnet       0.95      1.00      0.97      2919
  BruteForce       0.19      0.76      0.31      3962
         DoS       0.96      1.00      0.98     32642

    accuracy                           0.92    183247
   macro avg       0.77      0.91      0.80    183247
weighted avg       0.97      0.92      0.94    183247

  Training ExtraTrees + SMOTE... done in 532.5s
  Accuracy=0.8950  F1_macro=0.7822  F1_weighted=0.9178  AUC=0.9847
              precision    recall  f1-score   support

      Benign       0.99      0.87      0.93    143724
      Botnet       0.95      1.00      0.97      2919
  BruteForce       0.18      0.77      0.29      3962
         DoS       0.89      1.00      0.94     32642

    accuracy                           0.89   

## 📈 Cell 14 — Gradient Boosting (all strategies)
> ⚠️ Gradient Boosting is single-threaded and the slowest model — expect ~5-15 min per strategy.

In [ ]:
print('=' * 70)
print('GRADIENT BOOSTING  (slow — single threaded)')
print('=' * 70)

for strategy_name, (X_tr, y_tr) in strategies.items():
    label = f'GradientBoosting + {strategy_name}'
    model = GradientBoostingClassifier(
        n_estimators=100, max_depth=10, learning_rate=0.1,
        random_state=42
    )
    evaluate_model(model, X_tr, y_tr, X_test_scaled, y_test_enc, label)

print('\n✓ Gradient Boosting — all strategies done')

GRADIENT BOOSTING  (slow — single threaded)
  Training GradientBoosting + NoResample... 

## 🏆 Cell 15 — Results Comparison Table

In [ ]:
print('=' * 70)
print('RESULTS SUMMARY')
print('=' * 70)

if not results:
    print('ERROR: results dict is empty. Run model cells first.')
else:
    comparison = pd.DataFrame([
        {
            'Model': k,
            'Accuracy':          v['accuracy'],
            'F1_Macro':          v['f1_macro'],
            'F1_Weighted':       v['f1_weighted'],
            'Precision_Macro':   v['precision_macro'],
            'Recall_Macro':      v['recall_macro'],
            'ROC_AUC_Macro':     v['roc_auc_macro'] or 0.0,
        }
        for k, v in results.items()
    ])

    comparison = comparison.sort_values('F1_Weighted', ascending=False)
    pd.set_option('display.max_colwidth', 50)
    pd.set_option('display.float_format', '{:.4f}'.format)
    print(comparison.to_string(index=False))

    best_key = comparison.iloc[0]['Model']
    print(f'\n{"="*70}')
    print(f'🏆  BEST MODEL: {best_key}')
    print(f'    F1 Weighted : {comparison.iloc[0]["F1_Weighted"]:.4f}')
    print(f'    F1 Macro    : {comparison.iloc[0]["F1_Macro"]:.4f}')
    print(f'    Accuracy    : {comparison.iloc[0]["Accuracy"]:.4f}')
    print(f'{"="*70}')

## 🔲 Cell 16 — Confusion Matrix (Best Model)

In [ ]:
best_result = results[best_key]
cm = best_result['confusion_matrix']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title(f'Confusion Matrix (counts)\n{best_key}', fontsize=10)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Normalised
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[1])
axes[1].set_title('Confusion Matrix (normalised)', fontsize=10)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrix_best.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved confusion_matrix_best.png')

## 📌 Cell 17 — Feature Importance (Best Model)

In [ ]:
best_model = best_result['model']

if hasattr(best_model, 'feature_importances_'):
    fi_df = pd.DataFrame({
        'feature':    final_features,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)

    print('Top 20 features by importance:')
    print(fi_df.head(20).to_string(index=False))

    plt.figure(figsize=(10, 7))
    sns.barplot(data=fi_df.head(20), x='importance', y='feature', palette='viridis')
    plt.title(f'Top 20 Feature Importances\n{best_key}')
    plt.tight_layout()
    plt.savefig('feature_importance_best.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Saved feature_importance_best.png')
else:
    print('Best model does not expose feature_importances_ — skipping plot.')

## 🔄 Cell 18 — Cross-Validation (Best Model)

In [ ]:
print('=' * 70)
print('CROSS-VALIDATION — 5-Fold Stratified (Best Model)')
print('=' * 70)

best_strategy = best_key.split(' + ')[1]
X_best, y_best = strategies[best_strategy]

# Re-instantiate a fresh model (same params) — avoids data leakage from the
# already-fitted best_model
import copy
fresh_model = copy.deepcopy(best_model)
fresh_model.set_params(**{k: v for k, v in fresh_model.get_params().items()})

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'accuracy':        'accuracy',
    'f1_macro':        'f1_macro',
    'f1_weighted':     'f1_weighted',
    'precision_macro': 'precision_macro',
    'recall_macro':    'recall_macro',
}

print('Running 5-fold CV (may take several minutes)...')
t0 = time.time()
cv_results = cross_validate(
    fresh_model, X_best, y_best,
    cv=skf, scoring=scoring,
    n_jobs=-1, return_train_score=True
)
print(f'✓ CV done in {time.time()-t0:.1f}s\n')

for metric in ['accuracy', 'f1_macro', 'f1_weighted', 'precision_macro', 'recall_macro']:
    test_scores  = cv_results[f'test_{metric}']
    train_scores = cv_results[f'train_{metric}']
    print(f'{metric:20s}  test={test_scores.mean():.4f} ±{test_scores.std():.4f}'
          f'  train={train_scores.mean():.4f} ±{train_scores.std():.4f}')

## 📝 Cell 19 — Recommendations & Next Steps

In [ ]:
print('''
╔══════════════════════════════════════════════════════════════════════╗
║              RECOMMENDATIONS & NEXT STEPS                          ║
╠══════════════════════════════════════════════════════════════════════╣
║  1. HYPERPARAMETER TUNING                                           ║
║     • GridSearchCV / RandomizedSearchCV on best model               ║
║     • Key params: n_estimators, max_depth, learning_rate            ║
║     • Always use stratified CV                                      ║
║                                                                     ║
║  2. ENSEMBLE                                                        ║
║     • VotingClassifier with top 3 models (soft voting)              ║
║     • StackingClassifier with meta-learner                          ║
║                                                                     ║
║  3. COST-SENSITIVE LEARNING                                         ║
║     • Adjust class_weight: DoS/Botnet may cost more to miss         ║
║                                                                     ║
║  4. DEEP LEARNING                                                   ║
║     • LSTM for sequence-based features                              ║
║     • 1D-CNN for packet-level patterns                              ║
║                                                                     ║
║  5. INTERPRETABILITY                                                ║
║     • SHAP values for per-prediction explanations                   ║
║     • Per-class feature importance analysis                         ║
║                                                                     ║
║  6. DEPLOYMENT                                                      ║
║     • joblib.dump(best_model, "nids_model.pkl")                     ║
║     • Save scaler + label_encoder alongside the model               ║
║     • Monitor for concept/data drift in production                  ║
╚══════════════════════════════════════════════════════════════════════╝
''')
print('NOTEBOOK COMPLETED SUCCESSFULLY ✓')